# ¿Piloto o despliegue de precio?

Programa 09: el gate actual decide precio bajo capacidad/CVaR; esta extensión valora un experimento aleatorio antes de escalar. Sólo escenarios sintéticos.

Para cada brazo $p_j\sim Beta(a_j,b_j)$; $X_j|p_j\sim Binomial(n,p_j)$. Observado $x_j$, posterior $Beta(a_j+x_j,b_j+n-x_j)$.

$EVSI=\sum_{x_A,x_B}P(x_A)P(x_B)\max(0,m_AE[p_A|x_A],m_BE[p_B|x_B])-V_0$. La suma exacta debe tener masa uno y EVSI no negativa porque se puede ignorar la muestra. La decisión pilotear requiere EVSI > coste.

In [2]:
import json,sys
from pathlib import Path
CWD=Path.cwd().resolve()
ROOT=next((candidate for candidate in (CWD,CWD.parent) if (candidate/'src/evsi.py').exists()), CWD)
sys.path.insert(0,str(ROOT/'src'))
from evsi import Price,evaluate
p=[Price('A',100,11,9),Price('B',180,5,15)]
r=evaluate(p,20,1)
print(json.dumps(r,indent=2))

{
  "n_per_arm": 20,
  "pilot_cost": 1,
  "baseline_action": "A",
  "baseline_value": 55.00000000000001,
  "expected_value_after_sample": 57.21368047459871,
  "evsi": 2.2136804745987035,
  "net_evsi": 1.2136804745987035,
  "predictive_mass": 1.0000000000000044,
  "decision": "pilot"
}


## Baseline y sensibilidad
El baseline selecciona con medias previas. El piloto observa n clientes por brazo, actualiza dos tasas independientes, y elige precio o cerrar. El coste es una unidad sintética; no equivale a dinero real.

In [4]:
import csv
from pathlib import Path
CWD=Path.cwd().resolve()
ROOT=next((candidate for candidate in (CWD,CWD.parent) if (candidate/'results/sensitivity.csv').exists()), CWD)
rows=list(csv.DictReader(open(ROOT/'results/sensitivity.csv')))
[(r['n_per_arm'],r['pilot_cost'],round(float(r['net_evsi']),3),r['decision']) for r in rows if r['pilot_cost']=='1']

## Negativos y límites
Un coste demasiado alto rechaza el piloto; prior misspecification, interferencia o deriva invalidan la lectura del modelo. EVSI es valor ex ante bajo supuestos, no ventas incrementales observadas ni causalidad demostrada.

In [6]:
import sys
from pathlib import Path
CWD=Path.cwd().resolve()
ROOT=next((candidate for candidate in (CWD,CWD.parent) if (candidate/'src/evsi.py').exists()), CWD)
sys.path.insert(0,str(ROOT/'src'))
from evsi import Price,evaluate
r=evaluate([Price('A',100,11,9),Price('B',180,5,15)],20,1)
assert r['evsi'] >= -1e-10 and abs(r['predictive_mass']-1)<1e-10
print('OK: mass normalized; EVSI nonnegative; n=20 per arm')

OK: mass normalized; EVSI nonnegative; n=20 per arm
